# 03: Data Loaders - MLM & CLM

In [1]:
!uv pip install --quiet tokenizers datasets transformers torch

In [2]:
from datasets import Dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from transformers import PreTrainedTokenizerFast, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

## Dataset & Tokenizer

Same dataset and BPE tokenizer from notebooks 01 and 02.

In [3]:
dataset = Dataset.from_dict({"text": [
    "I play cricket.",
    "Yesterday, I was about to hit him.",
    "Is this even considered?",
    "You can't get my mobile number.",
]})

SPECIAL_TOKENS = ["[UNK]", "[PAD]", "[BOS]", "[EOS]", "[MASK]"]

_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
_tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
_tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(special_tokens=SPECIAL_TOKENS)
_tokenizer.train_from_iterator(dataset["text"], trainer)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=_tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[BOS]",
    eos_token="[EOS]",
    mask_token="[MASK]",
)

print(f"Vocab size: {tokenizer.vocab_size}")




Vocab size: 99


## 1. Masked Language Modeling (MLM)

Used by BERT-style models. `DataCollatorForLanguageModeling` randomly masks 15% of tokens and the model learns to predict them.

In [4]:
tokenized = dataset.map(lambda x: tokenizer(x["text"], truncation=True, max_length=32), batched=True)
tokenized = tokenized.remove_columns("text")
tokenized.set_format("torch")

mlm_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
mlm_loader = DataLoader(tokenized, batch_size=2, collate_fn=mlm_collator)

batch = next(iter(mlm_loader))
print("input_ids :", batch["input_ids"])
print("labels    :", batch["labels"])   # -100 = not masked (ignored in loss)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

input_ids : tensor([[ 9, 76, 96,  7,  1,  1,  1,  1,  1],
        [ 4,  6, 71,  4, 92, 86, 89, 88,  7]])
labels    : tensor([[-100, -100, -100, -100, -100, -100, -100, -100, -100],
        [  98, -100, -100,   77, -100, -100, -100, -100, -100]])


In [5]:
for i in range(len(batch["input_ids"])):
    input_ids = batch["input_ids"][i].tolist()
    labels    = batch["labels"][i].tolist()

    input_tokens = tokenizer.convert_ids_to_tokens(input_ids)
    label_tokens = [
        tokenizer.convert_ids_to_tokens([lid])[0] if lid != -100 else "·"
        for lid in labels
    ]

    print(f"Position : {list(range(len(input_tokens)))}")
    print(f"Input    : {input_tokens}")
    print(f"Labels   : {label_tokens}   ('·' = not masked, ignored in loss)")
    print()

Position : [0, 1, 2, 3, 4, 5, 6, 7, 8]
Input    : ['I', 'Ġplay', 'Ġcricket', '.', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Labels   : ['·', '·', '·', '·', '·', '·', '·', '·', '·']   ('·' = not masked, ignored in loss)

Position : [0, 1, 2, 3, 4, 5, 6, 7, 8]
Input    : ['[MASK]', ',', 'ĠI', '[MASK]', 'Ġabout', 'Ġto', 'Ġhit', 'Ġhim', '.']
Labels   : ['Yesterday', '·', '·', 'Ġwas', '·', '·', '·', '·', '·']   ('·' = not masked, ignored in loss)



### What you're seeing

Take sample 0 as an example:

```
Position : [ 0        1         2           3     4       5       6       7       8     ]
Input    : ['I',  '[MASK]', 'Ġcricket',  '.',  '[PAD]','[PAD]','[PAD]','[PAD]','[PAD]']
Labels   : ['·',  'Ġplay',     '·',      '·',   '·',    '·',    '·',    '·',    '·'  ]
```

**Step by step:**

1. **Original sentence:** `I play cricket.`  — tokenizes to `['I', 'Ġplay', 'Ġcricket', '.']`

2. **Padding:** The other sentence in this batch is longer, so this one is padded to match:
   `['I', 'Ġplay', 'Ġcricket', '.', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']`

3. **Masking (15% chance per token):** The collator randomly picked position 1 (`Ġplay`) to mask.  
   - Position 1 in `input` becomes `[MASK]`  
   - Position 1 in `labels` becomes `Ġplay` (the original token — the answer)  
   - All other positions in `labels` become `-100` (shown as `·`) — **ignored in loss**

4. **During training**, the model:
   - Reads the full input: `['I', '[MASK]', 'Ġcricket', '.', '[PAD]', ...]`
   - Produces a probability distribution over the vocabulary at **every** position
   - Loss is computed **only at position 1**, comparing the model's prediction to `Ġplay`
   - The model uses context from both sides (`I` on the left, `Ġcricket .` on the right) to guess the masked word — this is the **bidirectional** nature of MLM

**Why `-100`?**  
PyTorch's `CrossEntropyLoss` ignores any position where the label is `-100` by default. This is how the collator tells the loss function: *"only penalise the model for the masked positions, not the rest."*

## 2. Causal Language Modeling (CLM)

Used by GPT-style models. `DataCollatorForLanguageModeling` with `mlm=False` sets `labels = input_ids` (the model predicts every next token).

In [6]:
clm_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
clm_loader = DataLoader(tokenized, batch_size=2, collate_fn=clm_collator)

batch = next(iter(clm_loader))

for i in range(len(batch["input_ids"])):
    input_ids = batch["input_ids"][i].tolist()
    labels    = batch["labels"][i].tolist()

    input_tokens = tokenizer.convert_ids_to_tokens(input_ids)
    label_tokens = [
        tokenizer.convert_ids_to_tokens([lid])[0] if lid != -100 else "·"
        for lid in labels
    ]

    print(f"Position : {list(range(len(input_tokens)))}")
    print(f"Input    : {input_tokens}")
    print(f"Labels   : {label_tokens}   ('·' = padding, ignored in loss)")
    print()

Position : [0, 1, 2, 3, 4, 5, 6, 7, 8]
Input    : ['I', 'Ġplay', 'Ġcricket', '.', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Labels   : ['I', 'Ġplay', 'Ġcricket', '.', '·', '·', '·', '·', '·']   ('·' = padding, ignored in loss)

Position : [0, 1, 2, 3, 4, 5, 6, 7, 8]
Input    : ['Yesterday', ',', 'ĠI', 'Ġwas', 'Ġabout', 'Ġto', 'Ġhit', 'Ġhim', '.']
Labels   : ['Yesterday', ',', 'ĠI', 'Ġwas', 'Ġabout', 'Ġto', 'Ġhit', 'Ġhim', '.']   ('·' = padding, ignored in loss)



### What you're seeing

Take sample 0 (`I play cricket.`) as an example:

```
Position : [ 0      1          2          3      4       5       6       7       8    ]
Input    : ['I', 'Ġplay', 'Ġcricket',   '.',  '[PAD]','[PAD]','[PAD]','[PAD]','[PAD]']
Labels   : ['Ġplay','Ġcricket',  '.',  '[PAD]',  '·',    '·',    '·',    '·',   '·' ]
```

**Step by step:**

1. **Original sentence:** `I play cricket.` — tokenizes to `['I', 'Ġplay', 'Ġcricket', '.']`

2. **Padding:** Shorter sentence is padded to match the longest in the batch:
   `['I', 'Ġplay', 'Ġcricket', '.', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']`

3. **Labels = input shifted left by one:**  
   - Position 0 label = `Ġplay` → *"given `I`, predict `play`"*  
   - Position 1 label = `Ġcricket` → *"given `I play`, predict `cricket`"*  
   - Position 2 label = `.` → *"given `I play cricket`, predict `.`"*  
   - Position 3 label = `[PAD]` → first pad token  
   - Positions 4–8 label = `·` (`-100`) → pad-to-pad, ignored in loss

4. **During training**, the model:
   - Reads each token with a **causal mask** — position `t` can only attend to positions `0..t` (no peeking right)
   - At every position `t`, predicts what token comes next
   - Loss is computed at every **non-padding** position — unlike MLM where only masked positions contribute

**MLM vs CLM at a glance:**

| | MLM (BERT-style) | CLM (GPT-style) |
|---|---|---|
| Attention | Bidirectional (sees full sequence) | Unidirectional (left-to-right only) |
| Labels | Only masked positions (~15%) | Every non-padding position |
| Goal | Fill in blanks | Predict next token |
| Good for | Understanding tasks (classification, QA) | Generation tasks (chat, completion) |